# ZTF × MostHosts Cross-Match

This notebook merges two catalogs:

| Catalog | File | Key column |
|---|---|---|
| ZTF SN Ia photometric fits | `data/ZTF_snia_data.csv` | `ztfname` |
| MostHosts host-galaxy catalog | `data/mosthosts_20250408.csv` | `sn_name_sp` |

Rows are matched when `ztfname == sn_name_sp`.  
All ZTF columns are prefixed with `ztf_`, all MostHosts columns with `mosthost_`.  
The merged table is written to `data/ztf_mosthost_match.csv`.

> **Note:** MostHosts can list multiple candidate host galaxies per SN (`hostnum` 0, 1, 2 ...),  
> so the merged table will have more rows than the number of unique matched SNe.

## 0. Imports

In [ ]:
import pandas as pd

## 1. Load the catalogs

- `index_col=0` on the ZTF file drops the unnamed row-index column that pandas wrote when the CSV was originally saved.
- `low_memory=False` silences dtype-inference warnings caused by mixed-type columns in the large MostHosts file.

In [ ]:
ztf  = pd.read_csv('data/ZTF_snia_data.csv',      index_col=0, low_memory=False)
most = pd.read_csv('data/mosthosts_20250408.csv',              low_memory=False)

print(f'ZTF rows      : {len(ztf):,}  |  unique names     : {ztf["ztfname"].nunique():,}')
print(f'MostHosts rows: {len(most):,}  |  unique sn_name_sp: {most["sn_name_sp"].nunique():,}')

## 2. Check overlap before merging

How many ZTF SNe have a corresponding entry in MostHosts?

In [ ]:
ztf_names  = set(ztf['ztfname'])
most_names = set(most['sn_name_sp'])
overlap    = ztf_names & most_names

print(f'ZTF names found in MostHosts: {len(overlap):,} / {len(ztf_names):,}')
print(f'Sample matches              : {sorted(overlap)[:5]}')

## 3. Prefix columns and merge

Every ZTF column gets a `ztf_` prefix and every MostHosts column gets a `mosthost_` prefix  
**before** the join, so there is no column-name ambiguity in the merged output.

An **inner join** is used so only SNe present in both catalogs appear in the output.

In [ ]:
# prefix all columns before joining to avoid ambiguity
ztf.columns  = ['ztf_'      + c for c in ztf.columns]
most.columns = ['mosthost_' + c for c in most.columns]

# inner join: only keep SNe present in both catalogs
merged = pd.merge(
    ztf,
    most,
    left_on  = 'ztf_ztfname',         # ZTF name in the ZTF table
    right_on = 'mosthost_sn_name_sp', # ZTF name in the MostHosts table
    how      = 'inner'
)

print(f'Merged rows : {len(merged):,}')
print(f'Columns     : {len(merged.columns)}')
merged.head(3)

## 4. Save output

In [ ]:
out_path = 'data/ztf_mosthost_match.csv'
merged.to_csv(out_path, index=False)
print(f'Saved {len(merged):,} rows x {len(merged.columns)} columns -> {out_path}')

## 5. Check `mosthost_ls_id_dr9` coverage

Verify whether every row in the merged catalog has a Legacy Survey DR9 ID (`mosthost_ls_id_dr9`),
and display the value for every row.

In [ ]:
# Load the merged catalog from disk (so this cell can be run independently)
merged = pd.read_csv('data/ztf_mosthost_match.csv', low_memory=False)

# --- coverage check ---
total_rows  = len(merged)
missing     = merged['mosthost_ls_id_dr9'].isna().sum()
has_id      = total_rows - missing

print(f'Total rows             : {total_rows:,}')
print(f'Rows WITH  ls_id_dr9  : {has_id:,}')
print(f'Rows WITHOUT ls_id_dr9: {missing:,}')
print(f'Coverage               : {has_id / total_rows * 100:.1f}%')
print()

# --- print ls_id_dr9 for every row ---
# Display SN name, host number, and the DR9 ID side-by-side for easy reading
id_table = merged[['ztf_ztfname', 'mosthost_hostnum', 'mosthost_ls_id_dr9']].copy()
id_table.index = range(len(id_table))  # clean 0-based index
print(id_table.to_string())

## 6. Filter: drop rows missing `mosthost_ls_id_dr9`

Remove any row where `mosthost_ls_id_dr9` is NaN (i.e. no Legacy Survey DR9 match exists for that host)
and save the cleaned table to a new file.

In [ ]:
# Load the merged catalog from disk
merged = pd.read_csv('data/ztf_mosthost_match.csv', low_memory=False)

# Drop rows where mosthost_ls_id_dr9 is NaN
filtered = merged.dropna(subset=['mosthost_ls_id_dr9'])

print(f'Rows before filter: {len(merged):,}')
print(f'Rows after filter : {len(filtered):,}')
print(f'Rows dropped      : {len(merged) - len(filtered):,}')

# Save to new file
out_path = 'data/ztf_mosthost_match_filtered_ls_id.csv'
filtered.to_csv(out_path, index=False)
print(f'\nSaved -> {out_path}')